# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [1]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

CombinedModel(
  (block1): LinearReLUBlock(
    (linear): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
  )
  (block2): LinearTanhBlock(
    (linear): Linear(in_features=8, out_features=8, bias=True)
    (activation): Tanh()
  )
  (output): Linear(in_features=8, out_features=2, bias=True)
)


# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [2]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)

Registered parameters:
block1.linear.weight torch.Size([8, 4])
block1.linear.bias torch.Size([8])
block2.linear.weight torch.Size([8, 8])
block2.linear.bias torch.Size([8])
output.weight torch.Size([2, 8])
output.bias torch.Size([2])


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [3]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:", model.output.weight.grad is not None)
model.output.weight.grad


Gradient computed for output layer: True


tensor([[ 0.1677, -0.0875, -0.1337,  0.0982,  0.0915,  0.0133,  0.0582,  0.1256],
        [ 0.4729, -0.0595, -0.2310,  0.0898, -0.0282,  0.0248,  0.1084,  0.0062]])

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [4]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

Model device: cpu
Model dtype: torch.float32


### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [5]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

In [6]:
print(state_dict.keys())

odict_keys(['block1.linear.weight', 'block1.linear.bias', 'block2.linear.weight', 'block2.linear.bias', 'output.weight', 'output.bias'])


# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [7]:
import torch
import torch.nn as nn


# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)

Training mode output:
 tensor([[ 0.1565, -0.0662],
        [ 0.4698,  0.4866],
        [ 1.0366, -1.1867],
        [-0.1673,  0.5530],
        [-0.0938,  0.5942]], grad_fn=<AddmmBackward0>)
Evaluation mode output:
 tensor([[ 0.2067, -0.0203],
        [ 0.1641, -0.0246],
        [ 0.2588, -0.0193],
        [ 0.2222, -0.0060],
        [ 0.1134, -0.0723]])


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [8]:
import torch
import torch.nn as nn


class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)

Output no_grad method:
 tensor([[-0.2035,  0.6342],
        [ 0.4789,  0.0740],
        [ 0.5629, -0.1251],
        [ 0.8740, -0.3519],
        [-0.5361,  0.7014]])
Output inference_mode method:
 tensor([[-0.2035,  0.6342],
        [ 0.4789,  0.0740],
        [ 0.5629, -0.1251],
        [ 0.8740, -0.3519],
        [-0.5361,  0.7014]])
Output no_grad context manager:
 tensor([[-0.2035,  0.6342],
        [ 0.4789,  0.0740],
        [ 0.5629, -0.1251],
        [ 0.8740, -0.3519],
        [-0.5361,  0.7014]])
Output inference_mode context manager:
 tensor([[-0.2035,  0.6342],
        [ 0.4789,  0.0740],
        [ 0.5629, -0.1251],
        [ 0.8740, -0.3519],
        [-0.5361,  0.7014]])


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [9]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

fc.weight: requires_grad=False
fc.bias: requires_grad=False
Output shape: torch.Size([5, 2])


# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [10]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self


class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode in train()")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode in eval()")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)

CustomBlock set to train mode in train()
CustomBlock set to eval mode in eval()
CustomBlock set to eval mode in train()
Output training mode: tensor([[0.3783, 1.5313, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000]], grad_fn=<MulBackward0>)
Output eval mode: tensor([[0.1892, 0.7657, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 1.4128]])


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [11]:
seq_model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)

nn.Sequential output shape: torch.Size([5, 2])


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [12]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2)])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x


ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

nn.ModuleList output shape: torch.Size([5, 2])


## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [13]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({"branch1": nn.Linear(4, 8), "branch2": nn.Linear(4, 8)})
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)


md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

nn.ModuleDict output shape: torch.Size([5, 2])


## Type hints and Shape Information

For production code, a practical pattern is to use `Tensor` for type hints and document the expected shape next to the signature.

`Tensor` tells us what kind of object is expected, while `[B, C, H, W]` documents the tensor dimensions explicitly.


In [14]:
from torch import Tensor


def normalize_images(x: Tensor) -> Tensor:
    # x: [B, C, H, W]
    return x / 255.0


## Работа на семинаре

### LSTM

![lstm](assets/LSTM.png)

In [15]:
class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size) -> None:
        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size

        self.W_f = nn.Parameter(torch.randn(self.hidden_size, self.hidden_size + self.input_size))
        self.b_f = nn.Parameter(torch.zeros(self.hidden_size))

        self.W_c = nn.Parameter(torch.randn(self.hidden_size, self.hidden_size + self.input_size))
        self.b_c = nn.Parameter(torch.zeros(self.hidden_size))

        self.W_i = nn.Parameter(torch.randn(self.hidden_size, self.hidden_size + self.input_size))
        self.b_i = nn.Parameter(torch.zeros(self.hidden_size))

        self.W_o = nn.Parameter(torch.randn(self.hidden_size, self.hidden_size + self.input_size))
        self.b_o = nn.Parameter(torch.zeros(self.hidden_size))

    def forward(self, c: torch.Tensor, h: torch.Tensor, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        concat = torch.cat([h, x], dim=-1)

        # вероятность забыть (на самом деле сохранить информацию)
        forget_t = torch.sigmoid(concat @ self.W_f.T + self.b_f)
        candidate_t = torch.tanh(concat @ self.W_c.T + self.b_c)
        input_t = torch.sigmoid(concat @ self.W_i.T + self.b_i)  # вероятность запомнить

        cell_state_t = forget_t * c + input_t * candidate_t

        output_t = torch.sigmoid(concat @ self.W_o.T + self.b_o)  # вероятность выдать наружу
        h_t = output_t * torch.tanh(cell_state_t)

        return (h_t, cell_state_t)

### Inception

![inception](assets/inception.png)

In [16]:
class Inception(nn.Module):
    def __init__(self, in_channels, ch1x1, ch3x3_reduce, ch3x3, ch5x5_reduce, ch5x5, pool_proj):
        """
        in_channels - число каналов входного тензора

        ch1x1 - число выходных каналов после свертки 1x1

        ch3x3_reduce - число выходных каналов после свертки 1x1 перед сверткой 3x3
        ch3x3 - число выходных каналов после свертки 3x3

        ch5x5_reduce - число выходных каналов после свертки 1x1 перед сверткой 5x5
        ch5x5 - число выходных каналов после свертки 5x5

        pool_proj - - число выходных каналов после свертки 1x1 в ветке с max pooling
        """
        super().__init__()

        # первая ветка
        branch_1 = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=ch1x1, kernel_size=1),  # [B, ch1x1, H, W]
            nn.ReLU(),
        )

        # вторая ветка
        branch_2 = nn.Sequential(
            # [B, ch3x3_reduce, H, W]
            nn.Conv2d(in_channels=in_channels, out_channels=ch3x3_reduce, kernel_size=1),
            nn.ReLU(),
            # [B, ch3x3, H, W]
            nn.Conv2d(in_channels=ch3x3_reduce, out_channels=ch3x3, kernel_size=3, padding=1),
            nn.ReLU(),
        )

        # третья ветка
        branch_3 = nn.Sequential(
            # [B, ch5x5_reduce, H, W]
            nn.Conv2d(in_channels=in_channels, out_channels=ch5x5_reduce, kernel_size=1),
            nn.ReLU(),
            # [B, ch5x5, H, W]
            nn.Conv2d(in_channels=ch5x5_reduce, out_channels=ch5x5, kernel_size=5, padding=2),
            nn.ReLU(),
        )

        # четвертая ветка
        branch_4 = nn.Sequential(
            # [B, in_channels, H, W]
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            # [B, pool_proj, H, W]
            nn.Conv2d(in_channels=in_channels, out_channels=pool_proj, kernel_size=1),
            nn.ReLU(),
        )

        self.branches = nn.ModuleDict(
            {
                "branch_1": branch_1,
                "branch_2": branch_2,
                "branch_3": branch_3,
                "branch_4": branch_4,
            }
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.cat([branch(x) for branch in self.branches.values()], dim=1)

### SE

![se](assets/SqueezeAndExcite.png)

In [17]:
class SE(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden_size = max(1, channels // reduction)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, channels),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        weights = self.excitation(self.pool(x).flatten(1))
        return x * weights[:, :, None, None]

### Selective Kernel

![selective](assets/SelectiveKernel.png)


In [ ]:
class SelectiveKernel(nn.Module):
    def __init__(self, channels, reduction=16, groups=1):
        super().__init__()
        hidden_size = max(32, channels // reduction)
        self.branches = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Conv2d(channels, channels, kernel_size, padding=kernel_size // 2, groups=groups, bias=False),
                    nn.BatchNorm2d(channels),
                    nn.ReLU(),
                )
                for kernel_size in (3, 5)
            ]
        )
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden_size),
            nn.ReLU(),
        )
        self.attention = nn.Linear(hidden_size, 2 * channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = torch.stack([branch(x) for branch in self.branches], dim=1)
        pooled = features.sum(dim=1).mean(dim=(-2, -1))
        scores = self.attention(self.fc(pooled)).reshape(x.size(0), 2, x.size(1))
        weights = torch.softmax(scores, dim=1)
        return (features * weights[:, :, :, None, None]).sum(dim=1)

### PatchMerger

![patchmerger](assets/PatchMerger.png)


In [ ]:
class PatchMerger(nn.Module):
    def __init__(self, dim, num_patches):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.weight = nn.Parameter(torch.randn(num_patches, dim) / dim**0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.norm(x)
        scores = (x @ self.weight.T).transpose(-2, -1)
        weights = torch.softmax(scores, dim=-2)
        return weights @ x

## Домашнее задание

## Задание 1, реализуйте требуемые блоки

## ResNet Block (0.1 балл)

![Resnet](assets/ResBlock.png)

https://arxiv.org/pdf/1512.03385

In [20]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels=None, stride=1):
        super().__init__()
        out_channels = in_channels if out_channels is None else out_channels
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.shortcut = nn.Identity()
        if in_channels != out_channels or stride != 1:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.layers(x) + self.shortcut(x))

## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](assets/DepthWiseConv.png)

https://arxiv.org/pdf/1610.02357

In [ ]:
class SeparableConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size,
            stride=stride,
            padding=padding,
            groups=in_channels,
        )
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))

## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [22]:
from typing import Optional
import torch
from torch import nn
import numpy as np


class VanillaAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.align = nn.Linear(hidden_size, hidden_size, bias=False)
        self.value = nn.Linear(hidden_size, hidden_size, bias=False)
        self.query = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, query, key):
        scores = (key @ self.align(query).unsqueeze(-1)).squeeze(-1)
        weights = scores.softmax(dim=-1)
        context = (weights.unsqueeze(1) @ key).squeeze(1)
        return torch.tanh(self.value(context) + self.query(query))

## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [23]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, blocked_mask=None):
        scores = query @ key.transpose(-2, -1) / query.size(-1) ** 0.5
        if blocked_mask is not None:
            scores = scores.masked_fill(blocked_mask, -torch.inf)
            scores = scores.masked_fill(blocked_mask.all(dim=-1, keepdim=True), 0)
        weights = scores.softmax(dim=-1)
        if blocked_mask is not None:
            weights = weights.masked_fill(blocked_mask, 0)
        return self.dropout(weights) @ value

## Multihead Attention (0.1 балл)

![MultiheadAttention](assets/MultiheadAttention.webp)

https://arxiv.org/abs/1706.03762


In [24]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        if num_heads <= 0 or d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attention = ScaledDotProductAttention(dropout)

    def forward(self, query, key, value, blocked_mask=None):
        batch_size, query_length, d_model = query.shape
        query = self.q_proj(query).reshape(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        key = self.k_proj(key).reshape(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        value = self.v_proj(value).reshape(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        if blocked_mask is not None and blocked_mask.ndim == 3:
            blocked_mask = blocked_mask.unsqueeze(1)
        output = self.attention(query, key, value, blocked_mask)
        output = output.transpose(1, 2).reshape(batch_size, query_length, d_model)
        return self.out_proj(output)

## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](assets/TransformerEncoder.png)


https://arxiv.org/abs/1706.03762

In [25]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.dropout = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, blocked_mask=None):
        normalized = self.norm1(x)
        x = x + self.dropout(self.attention(normalized, normalized, normalized, blocked_mask))
        return x + self.mlp(self.norm2(x))

## MLP Mixer (0.1 балл)


![MLPMixer](assets/MLPMixer.png)


https://arxiv.org/abs/2105.01601

In [26]:
class MLPMixerBlock(nn.Module):
    def __init__(self, num_patches, channels, token_hidden_dim, channel_hidden_dim):
        super().__init__()
        self.norm1 = nn.LayerNorm(channels)
        self.token_mlp = nn.Sequential(
            nn.Linear(num_patches, token_hidden_dim),
            nn.GELU(),
            nn.Linear(token_hidden_dim, num_patches),
        )
        self.norm2 = nn.LayerNorm(channels)
        self.channel_mlp = nn.Sequential(
            nn.Linear(channels, channel_hidden_dim),
            nn.GELU(),
            nn.Linear(channel_hidden_dim, channels),
        )

    def forward(self, x):
        x = x + self.token_mlp(self.norm1(x).transpose(1, 2)).transpose(1, 2)
        return x + self.channel_mlp(self.norm2(x))

## ConvMixer (0.1 балл)

![ConvMixer](assets/ConvMixer.png)


https://arxiv.org/abs/2201.09792

In [27]:
class ConvMixer(nn.Module):
    def __init__(self, channels, kernel_size=9):
        super().__init__()
        self.depthwise = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size, padding="same", groups=channels),
            nn.ReLU(),
            nn.BatchNorm2d(channels),
        )
        self.pointwise = nn.Sequential(
            nn.Conv2d(channels, channels, 1),
            nn.ReLU(),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        x = x + self.depthwise(x)
        return self.pointwise(x)

## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---


Все три подхода смешивают информацию между патчами и между каналами. Атеншн делает это с помощью весов, зависящих от входа, MLP-Mixer через полносвязные слои, ConvMixer через свёртки. Так что Attention не является обязательным для получения хороших признаков изображения.

Пусть $X \in \mathbb{R}^{N \times C}$, где $N$ — число патчей, $C$ — число каналов. Общую идею смешивания можно записать так:

$$
Y = \sum_r A_r(X) X B_r,
$$

где $A_r$ смешивает позиции, а $B_r$ — каналы. Здесь опущены смещения, нормализации и residual-связи, нелинейности Mixer-моделей временно заменены тождественной функцией для простоты сравнения.

Для Multihead Attention каждое слагаемое соответствует одной голове:

$$
A_r(X) = \operatorname{softmax}\left(\frac{(XW_r^Q)(XW_r^K)^T}{\sqrt{d_k}}\right),
\qquad B_r = W_r^V W_r^O.
$$

$W_r^O$ — часть выходной проекции для этой головы

Для MLP-Mixer тогда $A = W_{t2}W_{t1}$ и $B = W_{c1}W_{c2}$. На самом деле между линейными слоями стоит GELU:

$$
U = W_{t2}\,\operatorname{GELU}(W_{t1}X),
\qquad Y = \operatorname{GELU}(UW_{c1})W_{c2}.
$$

Полную нелинейную модель нельзя точно свести к одному произведению $AXB$.

Для ConvMixer $A_c = K_c$ — разреженная матрица depthwise-свёртки канала $c$, а $B_c = e_c e_c^T W_p$, где $e_c$ выбирает этот канал, $W_p$ задаёт pointwise-свёртку. 

- Multihead Attention: связывает все позиции и выбирает связи по содержимому. Обычное полное внимание $O(N^2C)$ вычислений на смешивание токенов, дополнительно для учёта порядка используется позиционное кодирование
- MLP-Mixer: имеет глобальные связи и не вычисляет сходство каждой пары токенов. Но token-mixing слои привязаны к числу патчей, а качество сильно зависит от данных и регуляризации. При ширине token-MLP порядка N его стоимость тоже будет квадрат.
- ConvMixer: использует локальность и общие фильтры и работает с разными пространственными размерами. При фиксированных каналах и ядре стоимость - линия. Далёкие позиции взаимодействуют через несколько слоёв, а фильтры не меняются в зависимости от содержимого изображения.